# Temporal modeling experiment — V2 nonlinear comparator

Extend the frozen V2 experiment with one constrained nonlinear comparator: gradient-boosted decision trees. The purpose is to test whether nonlinearities and interactions improve temporal discrimination beyond the Ridge specification, using exactly the same V2 dataset, feature blocks, external folds, inner temporal validation, 0.5 threshold, and 2020 sensitivity.

No new data, target, or predictors are introduced.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

df = pd.read_csv('../data/processed/modeling_data_v2.csv')

## Define V2 features and common comparison sample

In [2]:
financial_v2_features = [
    'current_liquidity',
    'liability_ratio',
    'return_on_assets',
    'loss_current_year',
    'previous_loss_rate',
    'delta_current_liquidity',
    'delta_liability_ratio',
    'delta_return_on_assets',
]

sporting_features = [
    'division',
    'points_per_game',
    'goal_difference_per_game',
    'relative_league_position',
    'promoted',
    'relegated',
]

common_sample = df.dropna(
    subset=financial_v2_features + sporting_features
).copy()

common_sample.shape

(262, 42)

In [3]:
common_sample.groupby('fiscal_year')['loss_next_year'].agg(
    observations='count',
    losses='sum',
    loss_rate='mean',
)

,observations,losses,loss_rate
fiscal_year,,,
2017,31,16,0.516129
2018,32,17,0.531250
2019,32,23,0.718750
2020,33,15,0.454545
2021,35,15,0.428571
2022,35,14,0.400000
2023,33,24,0.727273
2024,31,20,0.645161


## Model specification

The Ridge specification is unchanged from V2. The nonlinear comparator is `GradientBoostingClassifier`, applied to the same financial-only and financial + sporting feature blocks.

To keep the nonlinear test deliberately constrained, only tree depth is tuned (`1`, `2`, or `3`). The number of trees, learning rate, and minimum leaf size are fixed. Tree models use the raw continuous variables; `division` is one-hot encoded and the binary variables are passed through unchanged.

In [4]:
financial_continuous = [
    'current_liquidity',
    'liability_ratio',
    'return_on_assets',
    'previous_loss_rate',
    'delta_current_liquidity',
    'delta_liability_ratio',
    'delta_return_on_assets',
]

financial_binary = ['loss_current_year']

full_continuous = financial_continuous + [
    'points_per_game',
    'goal_difference_per_game',
    'relative_league_position',
]

full_categorical = ['division']
full_binary = financial_binary + ['promoted', 'relegated']

C_GRID = [0.01, 0.1, 1, 10, 100]
DEPTH_GRID = [1, 2, 3]

In [5]:
def build_ridge_pipeline(model_type, C):
    if model_type == 'financial':
        preprocessor = ColumnTransformer(
            transformers=[
                ('continuous', StandardScaler(), financial_continuous),
                ('binary', 'passthrough', financial_binary),
            ],
            remainder='drop',
        )
    else:
        preprocessor = ColumnTransformer(
            transformers=[
                ('continuous', StandardScaler(), full_continuous),
                (
                    'division',
                    OneHotEncoder(
                        categories=[[1, 2, 3, 4]],
                        drop='first',
                        handle_unknown='ignore',
                    ),
                    full_categorical,
                ),
                ('binary', 'passthrough', full_binary),
            ],
            remainder='drop',
        )

    return Pipeline([
        ('preprocess', preprocessor),
        (
            'model',
            LogisticRegression(
                C=C,
                solver='liblinear',
                max_iter=1000,
            ),
        ),
    ])

In [6]:
def build_boosting_pipeline(model_type, max_depth):
    if model_type == 'financial':
        preprocessor = ColumnTransformer(
            transformers=[
                ('continuous', 'passthrough', financial_continuous),
                ('binary', 'passthrough', financial_binary),
            ],
            remainder='drop',
        )
    else:
        preprocessor = ColumnTransformer(
            transformers=[
                ('continuous', 'passthrough', full_continuous),
                (
                    'division',
                    OneHotEncoder(
                        categories=[[1, 2, 3, 4]],
                        drop='first',
                        handle_unknown='ignore',
                        sparse_output=False,
                    ),
                    full_categorical,
                ),
                ('binary', 'passthrough', full_binary),
            ],
            remainder='drop',
        )

    return Pipeline([
        ('preprocess', preprocessor),
        (
            'model',
            GradientBoostingClassifier(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=max_depth,
                min_samples_leaf=10,
                random_state=42,
            ),
        ),
    ])

## Temporal hyperparameter selection

For each external fold, hyperparameters are selected using only earlier feature years. Internal validation follows the same expanding-window design as V2. ROC-AUC is calculated separately in each internal validation year and averaged across those years.

Exact ties favor the more regularized Ridge value (smaller `C`) and the shallower boosting tree.

In [7]:
def select_c(train_data, model_type):
    years = sorted(train_data['fiscal_year'].unique())
    tuning_rows = []

    for C in C_GRID:
        fold_aucs = []
        validation_years = []

        for validation_year in years:
            inner_train = train_data[
                train_data['fiscal_year'] < validation_year
            ]
            inner_validation = train_data[
                train_data['fiscal_year'] == validation_year
            ]

            if inner_train['fiscal_year'].nunique() < 2:
                continue
            if inner_train['loss_next_year'].nunique() < 2:
                continue
            if inner_validation['loss_next_year'].nunique() < 2:
                continue

            model = build_ridge_pipeline(model_type, C)
            model.fit(inner_train, inner_train['loss_next_year'])

            probabilities = model.predict_proba(inner_validation)[:, 1]
            fold_aucs.append(
                roc_auc_score(
                    inner_validation['loss_next_year'],
                    probabilities,
                )
            )
            validation_years.append(validation_year)

        tuning_rows.append({
            'C': C,
            'mean_inner_roc_auc': np.mean(fold_aucs),
            'std_inner_roc_auc': np.std(fold_aucs, ddof=0),
            'validation_years': validation_years,
        })

    tuning = pd.DataFrame(tuning_rows).sort_values(
        ['mean_inner_roc_auc', 'C'],
        ascending=[False, True],
    ).reset_index(drop=True)

    return tuning.loc[0, 'C'], tuning

In [8]:
def select_depth(train_data, model_type):
    years = sorted(train_data['fiscal_year'].unique())
    tuning_rows = []

    for max_depth in DEPTH_GRID:
        fold_aucs = []
        validation_years = []

        for validation_year in years:
            inner_train = train_data[
                train_data['fiscal_year'] < validation_year
            ]
            inner_validation = train_data[
                train_data['fiscal_year'] == validation_year
            ]

            if inner_train['fiscal_year'].nunique() < 2:
                continue
            if inner_train['loss_next_year'].nunique() < 2:
                continue
            if inner_validation['loss_next_year'].nunique() < 2:
                continue

            model = build_boosting_pipeline(model_type, max_depth)
            model.fit(inner_train, inner_train['loss_next_year'])

            probabilities = model.predict_proba(inner_validation)[:, 1]
            fold_aucs.append(
                roc_auc_score(
                    inner_validation['loss_next_year'],
                    probabilities,
                )
            )
            validation_years.append(validation_year)

        tuning_rows.append({
            'max_depth': max_depth,
            'mean_inner_roc_auc': np.mean(fold_aucs),
            'std_inner_roc_auc': np.std(fold_aucs, ddof=0),
            'validation_years': validation_years,
        })

    tuning = pd.DataFrame(tuning_rows).sort_values(
        ['mean_inner_roc_auc', 'max_depth'],
        ascending=[False, True],
    ).reset_index(drop=True)

    return int(tuning.loc[0, 'max_depth']), tuning

## Expanding-window external experiment

The external folds remain defined by feature year: 2020 through 2024 in the main experiment. The persistence baseline, both Ridge models, and both boosting models are evaluated on the same test rows.

In [9]:
def run_experiment(data, first_test_year):
    prediction_rows = []
    ridge_tuning_rows = []
    boosting_tuning_rows = []

    test_years = [
        year
        for year in sorted(data['fiscal_year'].unique())
        if year >= first_test_year
    ]

    for test_year in test_years:
        train = data[data['fiscal_year'] < test_year].copy()
        test = data[data['fiscal_year'] == test_year].copy()

        baseline_predictions = test['loss_current_year'].astype(int)

        for row_index, prediction in zip(test.index, baseline_predictions):
            prediction_rows.append({
                'club': test.loc[row_index, 'club'],
                'fiscal_year': test_year,
                'target_year': test.loc[row_index, 'target_year'],
                'model': 'baseline',
                'y_true': int(test.loc[row_index, 'loss_next_year']),
                'score': float(prediction),
                'prediction': int(prediction),
            })

        for model_type, model_name in [
            ('financial', 'financial_v2_ridge'),
            ('full', 'financial_sporting_v2_ridge'),
        ]:
            best_C, tuning = select_c(train, model_type)

            for _, tuning_row in tuning.iterrows():
                ridge_tuning_rows.append({
                    'test_feature_year': test_year,
                    'model': model_name,
                    'C': tuning_row['C'],
                    'mean_inner_roc_auc': tuning_row['mean_inner_roc_auc'],
                    'std_inner_roc_auc': tuning_row['std_inner_roc_auc'],
                    'selected': tuning_row['C'] == best_C,
                    'validation_years': tuning_row['validation_years'],
                })

            model = build_ridge_pipeline(model_type, best_C)
            model.fit(train, train['loss_next_year'])

            probabilities = model.predict_proba(test)[:, 1]
            predictions = (probabilities >= 0.5).astype(int)

            for row_index, probability, prediction in zip(
                test.index,
                probabilities,
                predictions,
            ):
                prediction_rows.append({
                    'club': test.loc[row_index, 'club'],
                    'fiscal_year': test_year,
                    'target_year': test.loc[row_index, 'target_year'],
                    'model': model_name,
                    'y_true': int(test.loc[row_index, 'loss_next_year']),
                    'score': float(probability),
                    'prediction': int(prediction),
                })

        for model_type, model_name in [
            ('financial', 'financial_v2_boosting'),
            ('full', 'financial_sporting_v2_boosting'),
        ]:
            best_depth, tuning = select_depth(train, model_type)

            for _, tuning_row in tuning.iterrows():
                boosting_tuning_rows.append({
                    'test_feature_year': test_year,
                    'model': model_name,
                    'max_depth': int(tuning_row['max_depth']),
                    'mean_inner_roc_auc': tuning_row['mean_inner_roc_auc'],
                    'std_inner_roc_auc': tuning_row['std_inner_roc_auc'],
                    'selected': int(tuning_row['max_depth']) == best_depth,
                    'validation_years': tuning_row['validation_years'],
                })

            model = build_boosting_pipeline(model_type, best_depth)
            model.fit(train, train['loss_next_year'])

            probabilities = model.predict_proba(test)[:, 1]
            predictions = (probabilities >= 0.5).astype(int)

            for row_index, probability, prediction in zip(
                test.index,
                probabilities,
                predictions,
            ):
                prediction_rows.append({
                    'club': test.loc[row_index, 'club'],
                    'fiscal_year': test_year,
                    'target_year': test.loc[row_index, 'target_year'],
                    'model': model_name,
                    'y_true': int(test.loc[row_index, 'loss_next_year']),
                    'score': float(probability),
                    'prediction': int(prediction),
                })

    return (
        pd.DataFrame(prediction_rows),
        pd.DataFrame(ridge_tuning_rows),
        pd.DataFrame(boosting_tuning_rows),
    )

## Main experiment

In [10]:
predictions, ridge_tuning, boosting_tuning = run_experiment(
    common_sample,
    first_test_year=2020,
)

In [11]:
ridge_tuning[ridge_tuning['selected']].reset_index(drop=True)

,test_feature_year,model,C,mean_inner_roc_auc,std_inner_roc_auc,selected,validation_years
0,2020,financial_v2_ridge,1.0,0.893720,0.000000,True,[2019]
1,2020,financial_sporting_v2_ridge,100.0,0.801932,0.000000,True,[2019]
2,2021,financial_v2_ridge,0.1,0.731079,0.138486,True,"[2019, 2020]"
3,2021,financial_sporting_v2_ridge,100.0,0.671337,0.130596,True,"[2019, 2020]"
4,2022,financial_v2_ridge,1.0,0.697413,0.144106,True,"[2019, 2020, 2021]"
5,2022,financial_sporting_v2_ridge,100.0,0.687558,0.109071,True,"[2019, 2020, 2021]"
6,2023,financial_v2_ridge,1.0,0.695679,0.124836,True,"[2019, 2020, 2021, 2022]"
7,2023,financial_sporting_v2_ridge,100.0,0.663627,0.103152,True,"[2019, 2020, 2021, 2022]"
8,2024,financial_v2_ridge,1.0,0.696358,0.111665,True,"[2019, 2020, 2021, 2022, 2023]"
9,2024,financial_sporting_v2_ridge,1.0,0.662382,0.096043,True,"[2019, 2020, 2021, 2022, 2023]"


In [12]:
boosting_tuning[boosting_tuning['selected']].reset_index(drop=True)

,test_feature_year,model,max_depth,mean_inner_roc_auc,std_inner_roc_auc,selected,validation_years
0,2020,financial_v2_boosting,1,0.727053,0.000000,True,[2019]
1,2020,financial_sporting_v2_boosting,1,0.712560,0.000000,True,[2019]
2,2021,financial_v2_boosting,1,0.661675,0.065378,True,"[2019, 2020]"
3,2021,financial_sporting_v2_boosting,2,0.651932,0.007488,True,"[2019, 2020]"
4,2022,financial_v2_boosting,2,0.669192,0.026744,True,"[2019, 2020, 2021]"
5,2022,financial_sporting_v2_boosting,2,0.661288,0.014575,True,"[2019, 2020, 2021]"
6,2023,financial_v2_boosting,2,0.686418,0.037771,True,"[2019, 2020, 2021, 2022]"
7,2023,financial_sporting_v2_boosting,2,0.669436,0.018933,True,"[2019, 2020, 2021, 2022]"
8,2024,financial_v2_boosting,2,0.648208,0.083553,True,"[2019, 2020, 2021, 2022, 2023]"
9,2024,financial_sporting_v2_boosting,2,0.642956,0.055601,True,"[2019, 2020, 2021, 2022, 2023]"


## Performance by external feature year

In [13]:
yearly_results = (
    predictions
    .groupby(['fiscal_year', 'model'])
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'roc_auc': roc_auc_score(group['y_true'], group['score']),
            'balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

yearly_results

,fiscal_year,model,observations,roc_auc,balanced_accuracy,accuracy
0,2020,baseline,33.0,0.566667,0.566667,0.545455
1,2020,financial_sporting_v2_boosting,33.0,0.579630,0.538889,0.515152
2,2020,financial_sporting_v2_ridge,33.0,0.540741,0.533333,0.515152
3,2020,financial_v2_boosting,33.0,0.596296,0.544444,0.515152
4,2020,financial_v2_ridge,33.0,0.551852,0.538889,0.515152
5,2021,baseline,35.0,0.600000,0.600000,0.600000
6,2021,financial_sporting_v2_boosting,35.0,0.680000,0.600000,0.600000
7,2021,financial_sporting_v2_ridge,35.0,0.720000,0.716667,0.714286
8,2021,financial_v2_boosting,35.0,0.620000,0.533333,0.514286
9,2021,financial_v2_ridge,35.0,0.626667,0.533333,0.514286


## Principal temporal ROC-AUC summary

The principal result remains the simple mean of the external yearly AUCs.

In [14]:
temporal_auc_results = (
    yearly_results
    .groupby('model')['roc_auc']
    .agg(
        mean_roc_auc='mean',
        std_roc_auc='std',
        min_roc_auc='min',
        max_roc_auc='max',
        folds='count',
    )
    .reset_index()
)

temporal_auc_results

,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,baseline,0.644109,0.071292,0.566667,0.738095,5
1,financial_sporting_v2_boosting,0.611745,0.070560,0.537037,0.693878,5
2,financial_sporting_v2_ridge,0.654691,0.093776,0.540741,0.772727,5
3,financial_v2_boosting,0.597225,0.092793,0.495370,0.738095,5
4,financial_v2_ridge,0.660886,0.072612,0.551852,0.736364,5


## Pooled out-of-time performance — complementary

In [15]:
pooled_results = (
    predictions
    .groupby('model')
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'pooled_roc_auc': roc_auc_score(group['y_true'], group['score']),
            'pooled_balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'pooled_accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

pooled_results

,model,observations,pooled_roc_auc,pooled_balanced_accuracy,pooled_accuracy
0,baseline,167.0,0.632696,0.632696,0.634731
1,financial_sporting_v2_boosting,167.0,0.566528,0.544879,0.544910
2,financial_sporting_v2_ridge,167.0,0.630178,0.649094,0.652695
3,financial_v2_boosting,167.0,0.562428,0.534666,0.538922
4,financial_v2_ridge,167.0,0.625000,0.600403,0.604790


## Sensitivity analysis without feature year 2020

In [16]:
sample_without_2020 = common_sample[
    common_sample['fiscal_year'] != 2020
].copy()

(
    predictions_without_2020,
    ridge_tuning_without_2020,
    boosting_tuning_without_2020,
) = run_experiment(
    sample_without_2020,
    first_test_year=2021,
)

In [17]:
ridge_tuning_without_2020[
    ridge_tuning_without_2020['selected']
].reset_index(drop=True)

,test_feature_year,model,C,mean_inner_roc_auc,std_inner_roc_auc,selected,validation_years
0,2021,financial_v2_ridge,1.0,0.893720,0.000000,True,[2019]
1,2021,financial_sporting_v2_ridge,100.0,0.801932,0.000000,True,[2019]
2,2022,financial_v2_ridge,1.0,0.738527,0.155193,True,"[2019, 2021]"
3,2022,financial_sporting_v2_ridge,100.0,0.727633,0.074300,True,"[2019, 2021]"
4,2023,financial_v2_ridge,1.0,0.736115,0.126761,True,"[2019, 2021, 2022]"
5,2023,financial_sporting_v2_ridge,100.0,0.697107,0.074458,True,"[2019, 2021, 2022]"
6,2024,financial_v2_ridge,1.0,0.717596,0.114368,True,"[2019, 2021, 2022, 2023]"
7,2024,financial_sporting_v2_ridge,1.0,0.697294,0.071263,True,"[2019, 2021, 2022, 2023]"


In [18]:
boosting_tuning_without_2020[
    boosting_tuning_without_2020['selected']
].reset_index(drop=True)

,test_feature_year,model,max_depth,mean_inner_roc_auc,std_inner_roc_auc,selected,validation_years
0,2021,financial_v2_boosting,1,0.727053,0.000000,True,[2019]
1,2021,financial_sporting_v2_boosting,1,0.712560,0.000000,True,[2019]
2,2022,financial_v2_boosting,1,0.657693,0.069360,True,"[2019, 2021]"
3,2022,financial_sporting_v2_boosting,2,0.666377,0.006957,True,"[2019, 2021]"
4,2023,financial_v2_boosting,2,0.704988,0.084543,True,"[2019, 2021, 2022]"
5,2023,financial_sporting_v2_boosting,2,0.719761,0.075711,True,"[2019, 2021, 2022]"
6,2024,financial_v2_boosting,2,0.661264,0.105338,True,"[2019, 2021, 2022, 2023]"
7,2024,financial_sporting_v2_boosting,2,0.674080,0.102759,True,"[2019, 2021, 2022, 2023]"


In [19]:
yearly_results_without_2020 = (
    predictions_without_2020
    .groupby(['fiscal_year', 'model'])
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'roc_auc': roc_auc_score(group['y_true'], group['score']),
            'balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

yearly_results_without_2020

,fiscal_year,model,observations,roc_auc,balanced_accuracy,accuracy
0,2021,baseline,35.0,0.600000,0.600000,0.600000
1,2021,financial_sporting_v2_boosting,35.0,0.618333,0.591667,0.571429
2,2021,financial_sporting_v2_ridge,35.0,0.653333,0.591667,0.571429
3,2021,financial_v2_boosting,35.0,0.588333,0.566667,0.542857
4,2021,financial_v2_ridge,35.0,0.583333,0.558333,0.542857
5,2022,baseline,35.0,0.738095,0.738095,0.742857
6,2022,financial_sporting_v2_boosting,35.0,0.826531,0.797619,0.800000
7,2022,financial_sporting_v2_ridge,35.0,0.636054,0.583333,0.571429
8,2022,financial_v2_boosting,35.0,0.775510,0.750000,0.742857
9,2022,financial_v2_ridge,35.0,0.731293,0.714286,0.714286


In [20]:
temporal_auc_results_without_2020 = (
    yearly_results_without_2020
    .groupby('model')['roc_auc']
    .agg(
        mean_roc_auc='mean',
        std_roc_auc='std',
        min_roc_auc='min',
        max_roc_auc='max',
        folds='count',
    )
    .reset_index()
)

temporal_auc_results_without_2020

,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,baseline,0.663470,0.065405,0.600000,0.738095,4
1,financial_sporting_v2_boosting,0.656839,0.122148,0.537037,0.826531,4
2,financial_sporting_v2_ridge,0.669759,0.031986,0.636054,0.709091,4
3,financial_v2_boosting,0.642802,0.107191,0.530093,0.775510,4
4,financial_v2_ridge,0.679393,0.073044,0.583333,0.740909,4


In [21]:
pooled_results_without_2020 = (
    predictions_without_2020
    .groupby('model')
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'pooled_roc_auc': roc_auc_score(group['y_true'], group['score']),
            'pooled_balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'pooled_accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

pooled_results_without_2020

,model,observations,pooled_roc_auc,pooled_balanced_accuracy,pooled_accuracy
0,baseline,134.0,0.657983,0.657983,0.656716
1,financial_sporting_v2_boosting,134.0,0.599483,0.607456,0.611940
2,financial_sporting_v2_ridge,134.0,0.658657,0.633506,0.641791
3,financial_v2_boosting,134.0,0.581630,0.589715,0.597015
4,financial_v2_ridge,134.0,0.662924,0.638895,0.641791


## Final principal comparison

In [22]:
comparison = pd.concat([
    temporal_auc_results.assign(sample='main'),
    temporal_auc_results_without_2020.assign(sample='without_2020'),
], ignore_index=True)

comparison[
    ['sample', 'model', 'mean_roc_auc', 'std_roc_auc', 'min_roc_auc', 'max_roc_auc', 'folds']
]

,sample,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,main,baseline,0.644109,0.071292,0.566667,0.738095,5
1,main,financial_sporting_v2_boosting,0.611745,0.070560,0.537037,0.693878,5
2,main,financial_sporting_v2_ridge,0.654691,0.093776,0.540741,0.772727,5
3,main,financial_v2_boosting,0.597225,0.092793,0.495370,0.738095,5
4,main,financial_v2_ridge,0.660886,0.072612,0.551852,0.736364,5
5,without_2020,baseline,0.663470,0.065405,0.600000,0.738095,4
6,without_2020,financial_sporting_v2_boosting,0.656839,0.122148,0.537037,0.826531,4
7,without_2020,financial_sporting_v2_ridge,0.669759,0.031986,0.636054,0.709091,4
8,without_2020,financial_v2_boosting,0.642802,0.107191,0.530093,0.775510,4
9,without_2020,financial_v2_ridge,0.679393,0.073044,0.583333,0.740909,4


## Nonlinear incremental check

Positive values favor boosting over Ridge within the same feature block. The final column tests whether sporting variables add discrimination within the nonlinear model.

In [23]:
principal_wide = comparison.pivot(
    index='sample',
    columns='model',
    values='mean_roc_auc',
)

nonlinear_check = pd.DataFrame({
    'financial_boosting_minus_ridge': (
        principal_wide['financial_v2_boosting']
        - principal_wide['financial_v2_ridge']
    ),
    'full_boosting_minus_ridge': (
        principal_wide['financial_sporting_v2_boosting']
        - principal_wide['financial_sporting_v2_ridge']
    ),
    'sporting_increment_with_boosting': (
        principal_wide['financial_sporting_v2_boosting']
        - principal_wide['financial_v2_boosting']
    ),
})

nonlinear_check

,financial_boosting_minus_ridge,full_boosting_minus_ridge,sporting_increment_with_boosting
sample,,,
main,-0.063661,-0.042945,0.014520
without_2020,-0.036591,-0.012920,0.014037
